## Computer Vision - Part 4

### Object Detection

Object detection is a computer vision task that involves identifying and localizing objects within an image or video. It is a fundamental problem in the field of computer vision and has numerous applications, including surveillance, autonomous vehicles, and image search.

In this lab, we will load a pre-trained YOLO (You Only Look Once) model and run inference on an image to detect objects. YOLO is a popular object detection algorithm that is known for its speed and accuracy.

In [3]:
from ultralytics import YOLO
import cv2

#### Lab 1: Setting Up the Environment

In [2]:
img = cv2.imread("images/image1.jpg")

# Shape of the image (height, width, channels)
print("Shape of the image:", img.shape)
# Total number of pixels (height × width)
num_pixels = img.shape[0] * img.shape[1]
print("Number of Pixels:", num_pixels)

Shape of the image: (683, 1024, 3)
Number of Pixels: 699392


### Lab 2: Load a YOLO Model and Run Inference

In [3]:
model = YOLO("yolov8n.pt")
results = model(img)


0: 448x640 1 person, 3 bicycles, 1 car, 52.0ms
Speed: 5.6ms preprocess, 52.0ms inference, 6.5ms postprocess per image at shape (1, 3, 448, 640)


#### Lab 3: Reading Boxes and Classes

In [4]:
for r in results:
    for box in r.boxes:
        cls_id = int(box.cls[0]) # class index (e.g., 0 = person)
        conf = float(box.conf[0]) # confidence score (e.g., 0.87)
        x1, y1, x2, y2 = box.xyxy[0] # box corners in pixels
        label = model.names[cls_id] # human-readable class name
        print(f"{label}: {conf:.2f} box=[{x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f}]")

bicycle: 0.94 box=[245,328,472,568]
car: 0.89 box=[386,244,682,397]
person: 0.86 box=[260,181,411,518]
bicycle: 0.48 box=[79,267,117,315]
bicycle: 0.27 box=[78,268,102,316]


#### Lab 4: Computing the Box Center

In [5]:
# Loop through all detections
for r in results:
    for box in r.boxes:
        # Get coordinates
        x1, y1, x2, y2 = box.xyxy[0]
        # Convert to int
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        # Draw bounding box
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        # Compute center
        u = int((x1 + x2) / 2)
        v = int((y1 + y2) / 2)
        # Draw center point
        cv2.circle(img, (u, v), 5, (0, 0, 255), -1)
        # Show confidence and class label
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        label = model.names[cls_id]
        text = f"{label} {conf:.2f}"
        cv2.putText(img, text, (x1, y1 - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

# Save the image with detections
cv2.imwrite("images/image1_detections.jpg", img)

True

#### Lab 5: Real-Time Webcam Detection

Real-time object detection can be performed using a webcam. We will capture video frames from the webcam, run inference on each frame, and display the results in real-time.

In [ ]:
cap = cv2.VideoCapture(0) # 0 = default webcam

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run detection
    results = model(frame)

    # Loop through detections
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            x1, y1, x2, y2 = box.xyxy[0]
            # Convert to int
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            # Compute center
            u = int((x1 + x2) / 2)
            v = int((y1 + y2) / 2)

            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # Draw center point
            cv2.circle(frame, (u, v), 6, (0, 0, 255), -1)

            # Put label and confidence
            label = f"{cls_id}: {conf:.2f}"
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display frame
    cv2.imshow("YOLO Detection", frame)
    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 2 persons, 103.7ms
Speed: 3.9ms preprocess, 103.7ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 136.1ms
Speed: 2.0ms preprocess, 136.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 179.3ms
Speed: 1.4ms preprocess, 179.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 42.4ms
Speed: 1.3ms preprocess, 42.4ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 45.2ms
Speed: 1.3ms preprocess, 45.2ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 60.5ms
Speed: 1.1ms preprocess, 60.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 49.9ms
Speed: 1.6ms preprocess, 49.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 36.4ms
Speed: 1.2ms preprocess, 36.4ms inference, 0.4ms postprocess per image at s

: 

For videos, we can use OpenCV to read video files and process each frame similarly to how we process webcam frames. This allows us to perform object detection on pre-recorded videos as well.

In [11]:
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture("input_video.mp4")

# Get video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

In [12]:
# Create output video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output_video.mp4", fourcc, fps, (width, height))

In [13]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Draw center
            u = int((x1 + x2) / 2)
            v = int((y1 + y2) / 2)
            cv2.circle(frame, (u, v), 5, (0, 0, 255), -1)

            # Show confidence and class label
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = model.names[cls_id]
            text = f"{label} {conf:.2f}"
            cv2.putText(frame, text, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # Write frame to output video
    out.write(frame)

# Release resources
cap.release()
out.release()


0: 384x640 2 persons, 62.0ms
Speed: 10.9ms preprocess, 62.0ms inference, 12.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 33.8ms
Speed: 1.2ms preprocess, 33.8ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 38.7ms
Speed: 1.1ms preprocess, 38.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 37.4ms
Speed: 1.1ms preprocess, 37.4ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 33.1ms
Speed: 1.1ms preprocess, 33.1ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 30.3ms
Speed: 1.1ms preprocess, 30.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 30.7ms
Speed: 1.0ms preprocess, 30.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 30.8ms
Speed: 1.0ms preprocess, 30.8ms inference, 0.5ms postprocess per image at shape